In [72]:
!pip install openpyxl
import os
import pandas as pd
import numpy as np
import sys
import gc

Defaulting to user installation because normal site-packages is not writeable


In [73]:
gc.collect
df_saeb_2019 = pd.read_csv("2019/TS_ALUNO_9EF.csv",  encoding='latin-1', sep=";", dtype=str)

In [74]:
gc.collect()
df_saeb_2021 = pd.read_csv("2021/TS_ALUNO_9EF.csv", encoding='latin-1', sep=";", dtype=str)

In [75]:
gc.collect()
df_saeb_2023 = pd.read_csv("2023/TS_ALUNO_9EF.csv",  encoding='latin-1', sep=";", dtype=str)

### Pré-processamento dos dados
Esta seção descreve o pré-processamento realizado nos dados antes de submetê-los à análise. São descritos as motivações e os processos relacionados à fusão de conjuntos e à limpeza de dados. 

#### 1. Filtro das colunas
Nessa etapa, serão retiradas de todos os datasets aquelas colunas que carregam informações que não precisam ser analisadas, ou seja, colunas consideradas desnecessárias aos objetivos do estudo. 

##### Variáveis referentes à avaliação:
Essas informações referem-se à estrutura e à aplicação da prova, não sendo essenciais para os objetivos deste trabalho, que não envolvem análise de participação, padrão de respostas, etc. Assim, optou-se por suas exclusões:

IN_PREENCHIMENTO_LP, IN_PREENCHIMENTO_MT, IN_PREENCHIMENTO_CH, IN_PREENCHIMENTO_CN (Indicadores de preenchimento da prova);

IN_PRESENCA_LP, IN_PRESENCA_MT, IN_PRESENCA_CH, IN_PRESENCA_CN (Indicadores de presença na prova);

ID_CADERNO_LP, ID_CADERNO_MT, ID_CADERNO_CH, ID_CADERNO_CN (Números dos cadernos de prova);

ID_BLOCO_1_LP, ID_BLOCO_2_LP, ID_BLOCO_1_MT, ID_BLOCO_2_MT, ID_BLOCO_1_CH, ID_BLOCO_2_CH, ID_BLOCO_3_CH, ID_BLOCO_1_CN, ID_BLOCO_2_CN, ID_BLOCO_3_CN (Identificadores dos blocos de prova);

NU_BLOCO_1_ABERTA_CH, NU_BLOCO_2_ABERTA_CH, NU_BLOCO_1_ABERTA_CN, NU_BLOCO_2_ABERTA_CN (Identificadores dos blocos de resposta construída da prova);  

TX_RESP_BLOCO1_LP, TX_RESP_BLOCO2_LP, TX_RESP_BLOCO1_MT, TX_RESP_BLOCO2_MT, TX_RESP_BLOCO1_CH, TX_RESP_BLOCO2_CH, TX_RESP_BLOCO3_CH, TX_RESP_BLOCO1_CN, TX_RESP_BLOCO2_CN, TX_RESP_BLOCO3_CN (Respostas do aluno a cada bloco da prova);

CO_CONCEITO_Q1_CH, CO_CONCEITO_Q2_CH, CO_CONCEITO_Q1_CN, CO_CONCEITO_Q2_CN (Conceitos obtidos nas questões de resposta construída);

IN_AMOSTRA (Indicador de participação da amostra)	

ESTRATO (Descrição dos estratos - Os estratos são compostos por características da participação da escola na avaliação e representam agrupamentos para os quais a avaliação fornece resultados confiáveis para Língua portuguesa e Matemática), ESTRATO_CIENCIAS (Descrição dos estratos para amostra de Ciências Humanas e Ciências da Natureza	- Os estratos são compostos por características da participação da escola na avaliação e representam agrupamentos para os quais a avaliação fornece resultados confiáveis para Ciências Humanas e Ciências da Natureza);

IN_PREENCHIMENTO_QUESTIONARIO (Indicador de preenchimento do questionário).

##### Variáveis referentes à proficiência em outras disciplinas:
Essas variáveis referem-se à proficiência em Língua Portuguesa, Ciências Humanas e Ciências da Natureza, ou seja, disciplinas que não são objeto do estudo.

IN_PROFICIENCIA_LP, IN_PROFICIENCIA_CH, IN_PROFICIENCIA_CN (Indicadores para cálculo da proficiência (no mínimo três itens respondidos no caderno de provas));	

PESO_ALUNO_LP, PESO_ALUNO_CH, PESO_ALUNO_CN (Peso do aluno na prova);

PROFICIENCIA_LP, PROFICIENCIA_CH, PROFICIENCIA_CN (Proficiência do aluno calculada na escala única do SAEB, com média = 0 e desvio = 1 na população de referência);

ERRO_PADRAO_LP, ERRO_PADRAO_CH, ERRO_PADRAO_CN (Erro padrão da proficiência);

PROFICIENCIA_LP_SAEB, PROFICIENCIA_CH_SAEB, PROFICIENCIA_CN_SAEB (Proficiência do aluno transformada na escala única do SAEB, com média = 250, desvio = 50(do SAEB/97));

ERRO_PADRAO_LP_SAEB, ERRO_PADRAO_CH_SAEB,  ERRO_PADRAO_CN_SAEB (Erro padrão da proficiência transformada).

##### Outras:
Os datasets dos diferentes anos possuem algumas variáveis diferentes entre si, portanto, algumas colunas devem ser desconsideradas por estarem presentes apenas na base de algum(ns) dos anos, o que comprometeria a consistência das análises multianuais.

IN_INSE (Indicador para cálculo do INSE (Indicador de Nível Socioeconômico));

INSE_ALUNO (Resultado individual do INSE para o aluno);

NU_TIPO_NIVEL_INSE (Classificação do Indicador de Nível Socioeconômico em 8 Grupos);

PESO_ALUNO_INSE (Peso do Aluno para cálculo do INSE).

##### Questionário:
Os datasets dos diferentes anos possuem algumas questões diferentes entre si, portanto, algumas colunas devem ser desconsideradas por estarem presentes apenas na base de algum(ns) dos anos, o que comprometeria a consistência das análises multianuais. 

Em 2019, serão removidas as seguintes questões (colunas) que diferem entre os anos: 
TX_RESP_Q003c, TX_RESP_Q003d, TX_RESP_Q007, TX_RESP_Q009b, TX_RESP_Q018a, TX_RESP_Q018b, TX_RESP_Q018c.

In [76]:
df_2019_red = df_saeb_2019.drop(columns=[
    'IN_PREENCHIMENTO_LP', 'IN_PREENCHIMENTO_MT', 'IN_PREENCHIMENTO_CH', 'IN_PREENCHIMENTO_CN', 'IN_PRESENCA_LP', 'IN_PRESENCA_MT', 'IN_PRESENCA_CH',
    'IN_PRESENCA_CN', 'ID_CADERNO_LP', 'ID_BLOCO_1_LP', 'ID_BLOCO_2_LP', 'ID_CADERNO_MT', 'ID_BLOCO_1_MT', 'ID_BLOCO_2_MT', 'ID_CADERNO_CH', 
    'ID_BLOCO_1_CH', 'ID_BLOCO_2_CH', 'ID_BLOCO_3_CH', 'NU_BLOCO_1_ABERTA_CH', 'NU_BLOCO_2_ABERTA_CH', 'ID_CADERNO_CN', 'ID_BLOCO_1_CN', 'ID_BLOCO_2_CN',
    'ID_BLOCO_3_CN', 'NU_BLOCO_1_ABERTA_CN', 'NU_BLOCO_2_ABERTA_CN', 'TX_RESP_BLOCO1_LP', 'TX_RESP_BLOCO2_LP', 'TX_RESP_BLOCO1_MT', 'TX_RESP_BLOCO2_MT',
    'TX_RESP_BLOCO1_CH', 'TX_RESP_BLOCO2_CH', 'TX_RESP_BLOCO3_CH', 'CO_CONCEITO_Q1_CH', 'CO_CONCEITO_Q2_CH', 'TX_RESP_BLOCO1_CN', 'TX_RESP_BLOCO2_CN', 
    'TX_RESP_BLOCO3_CN', 'CO_CONCEITO_Q1_CN', 'CO_CONCEITO_Q2_CN', 'IN_AMOSTRA', 'ESTRATO', 'ESTRATO_CIENCIAS', 'IN_PREENCHIMENTO_QUESTIONARIO',
    
    'IN_PROFICIENCIA_LP', 'IN_PROFICIENCIA_CH', 'IN_PROFICIENCIA_CN', 'PESO_ALUNO_LP', 'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_LP_SAEB', 
    'ERRO_PADRAO_LP_SAEB', 'PESO_ALUNO_CH', 'PROFICIENCIA_CH', 'ERRO_PADRAO_CH', 'PROFICIENCIA_CH_SAEB', 'ERRO_PADRAO_CH_SAEB', 'PESO_ALUNO_CN', 
    'PROFICIENCIA_CN', 'ERRO_PADRAO_CN', 'PROFICIENCIA_CN_SAEB', 'ERRO_PADRAO_CN_SAEB', 
    
    'TX_RESP_Q003C', 'TX_RESP_Q003D', 'TX_RESP_Q007', 'TX_RESP_Q009B', 'TX_RESP_Q018A', 'TX_RESP_Q018B', 'TX_RESP_Q018C'
])
df_2019_red

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,ID_TURMA,ID_SERIE,...,TX_RESP_Q013,TX_RESP_Q014,TX_RESP_Q015,TX_RESP_Q016,TX_RESP_Q017A,TX_RESP_Q017B,TX_RESP_Q017C,TX_RESP_Q017D,TX_RESP_Q017E,TX_RESP_Q019
0,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,C,A,B,A,D,A,D,B,A,C
1,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,B,A,A,A,D,A,D,B,A,C
2,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,B,A,A,A,D,A,D,C,A,D
3,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,C,A,A,A,C,A,D,C,A,D
4,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,C,A,A,A,C,A,*,B,A,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2388926,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,.,.,.,.,.,.,.,.,.,.
2388927,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,A,C,A,A,D,B,C,D,A,A
2388928,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,A,A,B,A,D,A,A,C,A,C
2388929,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,C,A,B,A,D,A,B,B,A,B


Em 2021, serão removidas as seguintes questões (colunas) que diferem entre os anos: TX_RESP_Q01, TX_RESP_Q02, TX_RESP_Q05, TX_RESP_Q06c, TX_RESP_Q06d, TX_RESP_Q09a, TX_RESP_Q11b, TX_RESP_Q11h, TX_RESP_Q15, TX_RESP_Q22a, TX_RESP_Q22b, TX_RESP_Q22c, TX_RESP_Q22d, TX_RESP_Q22e, TX_RESP_Q22f, TX_RESP_Q22g, TX_RESP_Q22h, TX_RESP_Q22i.

In [77]:
df_2021_red = df_saeb_2021.drop(columns=[
    'IN_PREENCHIMENTO_LP', 'IN_PREENCHIMENTO_MT', 'IN_PREENCHIMENTO_CH', 'IN_PREENCHIMENTO_CN', 'IN_PRESENCA_LP', 'IN_PRESENCA_MT', 'IN_PRESENCA_CH',
    'IN_PRESENCA_CN', 'ID_CADERNO_LP', 'ID_BLOCO_1_LP', 'ID_BLOCO_2_LP', 'ID_CADERNO_MT', 'ID_BLOCO_1_MT', 'ID_BLOCO_2_MT', 'ID_CADERNO_CH', 
    'ID_BLOCO_1_CH', 'ID_BLOCO_2_CH', 'ID_BLOCO_3_CH', 'NU_BLOCO_1_ABERTA_CH', 'NU_BLOCO_2_ABERTA_CH', 'ID_CADERNO_CN', 'ID_BLOCO_1_CN', 'ID_BLOCO_2_CN',
    'ID_BLOCO_3_CN', 'NU_BLOCO_1_ABERTA_CN', 'NU_BLOCO_2_ABERTA_CN', 'TX_RESP_BLOCO1_LP', 'TX_RESP_BLOCO2_LP', 'TX_RESP_BLOCO1_MT', 'TX_RESP_BLOCO2_MT',
    'TX_RESP_BLOCO1_CH', 'TX_RESP_BLOCO2_CH', 'TX_RESP_BLOCO3_CH', 'CO_CONCEITO_Q1_CH', 'CO_CONCEITO_Q2_CH', 'TX_RESP_BLOCO1_CN', 'TX_RESP_BLOCO2_CN', 
    'TX_RESP_BLOCO3_CN', 'CO_CONCEITO_Q1_CN', 'CO_CONCEITO_Q2_CN', 'IN_AMOSTRA', 'ESTRATO', 'ESTRATO_CIENCIAS', 'IN_PREENCHIMENTO_QUESTIONARIO','IN_INSE', 
    'INSE_ALUNO', 'NU_TIPO_NIVEL_INSE', 'PESO_ALUNO_INSE',

    'IN_PROFICIENCIA_LP', 'IN_PROFICIENCIA_CH', 'IN_PROFICIENCIA_CN', 'PESO_ALUNO_LP', 'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_LP_SAEB', 
    'ERRO_PADRAO_LP_SAEB', 'PESO_ALUNO_CH', 'PROFICIENCIA_CH', 'ERRO_PADRAO_CH', 'PROFICIENCIA_CH_SAEB', 'ERRO_PADRAO_CH_SAEB', 'PESO_ALUNO_CN', 
    'PROFICIENCIA_CN', 'ERRO_PADRAO_CN', 'PROFICIENCIA_CN_SAEB', 'ERRO_PADRAO_CN_SAEB', 
    
    'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q05','TX_RESP_Q06c', 'TX_RESP_Q06d', 'TX_RESP_Q09a', 
    'TX_RESP_Q11b', 'TX_RESP_Q11h', 'TX_RESP_Q15', 'TX_RESP_Q22a', 'TX_RESP_Q22b', 'TX_RESP_Q22c', 'TX_RESP_Q22d', 'TX_RESP_Q22e', 'TX_RESP_Q22f', 
    'TX_RESP_Q22g', 'TX_RESP_Q22h', 'TX_RESP_Q22i'
])
df_2021_red

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,ID_TURMA,ID_SERIE,...,TX_RESP_Q16,TX_RESP_Q17,TX_RESP_Q18,TX_RESP_Q19,TX_RESP_Q20a,TX_RESP_Q20b,TX_RESP_Q20c,TX_RESP_Q20d,TX_RESP_Q20e,TX_RESP_Q21
0,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,A,A,A,A,C,B,D,D,B,C
1,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,B,C,C,A,B,D,D,D,C,B
2,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,.,.,.,.,.,.,.,.,.,.
3,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,.,.,.,.,.,.,.,.,.,.
4,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,C,A,.,A,B,*,A,A,D,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2591932,2021,5,53,6322169,1,61397110,0,1,1475064,9,...,.,.,.,.,.,.,.,.,.,.
2591933,2021,5,53,6322169,1,61397110,0,1,1475064,9,...,.,.,.,.,.,.,.,.,.,.
2591934,2021,5,53,6322169,1,61397110,0,1,1475064,9,...,.,.,.,.,.,.,.,.,.,.
2591935,2021,5,53,6322169,1,61397110,0,1,1475064,9,...,.,.,.,.,.,.,.,.,.,.


Em 2023, serão removidas as seguintes questões (colunas) que diferem entre os anos: TX_RESP_Q01, TX_RESP_Q02, TX_RESP_Q05a, TX_RESP_Q05b, TX_RESP_Q05c, TX_RESP_Q06, TX_RESP_Q07c, TX_RESP_Q07d, TX_RESP_Q10a, TX_RESP_Q12g, TX_RESP_Q15a, TX_RESP_Q15b, TX_RESP_Q22a, TX_RESP_Q22b, TX_RESP_Q22c, TX_RESP_Q22d, TX_RESP_Q22e, TX_RESP_Q22f, TX_RESP_Q22g, TX_RESP_Q22h, TX_RESP_Q23a, TX_RESP_Q23b, TX_RESP_Q23c, TX_RESP_Q23d, TX_RESP_Q23e, TX_RESP_Q23f, TX_RESP_Q23g, TX_RESP_Q23h, TX_RESP_Q23i.

In [78]:
df_2023_red = df_saeb_2023.drop(columns=[
    'IN_PREENCHIMENTO_LP', 'IN_PREENCHIMENTO_MT', 'IN_PREENCHIMENTO_CH', 'IN_PREENCHIMENTO_CN', 'IN_PRESENCA_LP', 'IN_PRESENCA_MT', 'IN_PRESENCA_CH',
    'IN_PRESENCA_CN', 'ID_CADERNO_LP', 'ID_BLOCO_1_LP', 'ID_BLOCO_2_LP', 'ID_CADERNO_MT', 'ID_BLOCO_1_MT', 'ID_BLOCO_2_MT', 'ID_CADERNO_CH', 
    'ID_BLOCO_1_CH', 'ID_BLOCO_2_CH', 'NU_BLOCO_1_ABERTA_CH', 'NU_BLOCO_2_ABERTA_CH', 'ID_CADERNO_CN', 'ID_BLOCO_1_CN', 'ID_BLOCO_2_CN',
    'ID_BLOCO_3_CN', 'NU_BLOCO_1_ABERTA_CN', 'NU_BLOCO_2_ABERTA_CN', 'TX_RESP_BLOCO1_LP', 'TX_RESP_BLOCO2_LP', 'TX_RESP_BLOCO1_MT', 'TX_RESP_BLOCO2_MT',
    'TX_RESP_BLOCO1_CH', 'TX_RESP_BLOCO2_CH', 'CO_CONCEITO_Q1_CH', 'CO_CONCEITO_Q2_CH', 'TX_RESP_BLOCO1_CN', 'TX_RESP_BLOCO2_CN', 
    'TX_RESP_BLOCO3_CN', 'CO_CONCEITO_Q1_CN', 'CO_CONCEITO_Q2_CN', 'IN_AMOSTRA', 'ESTRATO', 'ESTRATO_CIENCIAS', 'IN_PREENCHIMENTO_QUESTIONARIO',
    'IN_INSE', 'INSE_ALUNO', 'NU_TIPO_NIVEL_INSE', 'PESO_ALUNO_INSE', 
    
    'IN_PROFICIENCIA_LP', 'IN_PROFICIENCIA_CH', 'IN_PROFICIENCIA_CN', 'PESO_ALUNO_LP', 'PROFICIENCIA_LP', 'ERRO_PADRAO_LP', 'PROFICIENCIA_LP_SAEB', 
    'ERRO_PADRAO_LP_SAEB', 'PESO_ALUNO_CH', 'PROFICIENCIA_CH', 'ERRO_PADRAO_CH', 'PROFICIENCIA_CH_SAEB', 'ERRO_PADRAO_CH_SAEB', 'PESO_ALUNO_CN', 
    'PROFICIENCIA_CN', 'ERRO_PADRAO_CN', 'PROFICIENCIA_CN_SAEB', 'ERRO_PADRAO_CN_SAEB', 
    
    'TX_RESP_Q01', 'TX_RESP_Q02', 'TX_RESP_Q05a', 'TX_RESP_Q05b', 'TX_RESP_Q05c', 
    'TX_RESP_Q06', 'TX_RESP_Q07c', 'TX_RESP_Q07d', 'TX_RESP_Q10a', 'TX_RESP_Q12g', 'TX_RESP_Q15a', 'TX_RESP_Q15b', 'TX_RESP_Q22a', 'TX_RESP_Q22b', 
    'TX_RESP_Q22c', 'TX_RESP_Q22d', 'TX_RESP_Q22e', 'TX_RESP_Q22f', 'TX_RESP_Q22g', 'TX_RESP_Q22h', 'TX_RESP_Q23a', 'TX_RESP_Q23b', 'TX_RESP_Q23c', 
    'TX_RESP_Q23d', 'TX_RESP_Q23e', 'TX_RESP_Q23f', 'TX_RESP_Q23g', 'TX_RESP_Q23h', 'TX_RESP_Q23i'
])
df_2023_red

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,ID_TURMA,ID_SERIE,...,TX_RESP_Q17,TX_RESP_Q18,TX_RESP_Q19,TX_RESP_Q20,TX_RESP_Q21a,TX_RESP_Q21b,TX_RESP_Q21c,TX_RESP_Q21d,TX_RESP_Q21e,TX_RESP_Q24
0,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,C,A,A,A,A,A,D,D,D,B
1,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,.,.,.,.,.,.,.,.,.,.
2,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,.,.,.,.,.,.,.,.,.,.
3,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,C,A,A,A,.,.,.,.,D,A
4,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,.,.,.,.,.,.,.,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2502902,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,C,A,A,A,B,C,A,A,D,C
2502903,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,A,C,A,A,C,D,D,C,D,C
2502904,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,.,.,.,.,.,.,.,.,.,.
2502905,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,.,.,A,A,B,A,B,A,D,C


#### 2. Remoção de dados que não são objetos do estudo
Como o objetivo do estudo é analisar o desempenho em matemática dos alunos de escolas públicas, serão selecionados apenas os dados desses alunos (a variável IN_PUBLICA é responsável por carregar essa informação). Dentro desse conjunto, é necessário avaliar a participação de cada aluno, e se ele cumpriu o requisito mínimo necessário para ter seu desempenho avaliado (a variável IN_PROFICIENCIA_MT, que é o indicador para cálculo da proficiência em Matemática - no mínimo três itens respondidos no caderno de provas de Língua Portuguesa e Matemática - engloba essas informações). 
Registros além desses não contribuem para as análises propostas, portanto, será aplicado um filtro para identificar e selecionar apenas os registros de interesse, garantindo que o dataset final contenha apenas participantes de escolas públicas que efetivamente tiveram seu desempenho avaliado.

##### 2019

In [79]:
df_2019_red = df_2019_red[(df_2019_red['IN_PROFICIENCIA_MT'] == '1') & (df_2019_red['IN_PUBLICA'] == '1')].copy()
df_2019_red

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,ID_TURMA,ID_SERIE,...,TX_RESP_Q013,TX_RESP_Q014,TX_RESP_Q015,TX_RESP_Q016,TX_RESP_Q017A,TX_RESP_Q017B,TX_RESP_Q017C,TX_RESP_Q017D,TX_RESP_Q017E,TX_RESP_Q019
0,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,C,A,B,A,D,A,D,B,A,C
1,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,B,A,A,A,D,A,D,B,A,C
2,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,B,A,A,A,D,A,D,C,A,D
3,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,C,A,A,A,C,A,D,C,A,D
4,2019,1,11,6311030,2,61255632,1,1,1030927,9,...,C,A,A,A,C,A,*,B,A,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2388923,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,B,A,B,A,D,B,*,B,A,C
2388924,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,A,A,C,A,D,A,B,B,B,A
2388927,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,A,C,A,A,D,B,C,D,A,A
2388928,2019,5,53,6316599,1,61322272,1,1,1067040,9,...,A,A,B,A,D,A,A,C,A,C


##### 2021

In [80]:
df_2021_red = df_2021_red[(df_2021_red['IN_PROFICIENCIA_MT'] == '1') & (df_2021_red['IN_PUBLICA'] == '1')].copy()
df_2021_red

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,ID_TURMA,ID_SERIE,...,TX_RESP_Q16,TX_RESP_Q17,TX_RESP_Q18,TX_RESP_Q19,TX_RESP_Q20a,TX_RESP_Q20b,TX_RESP_Q20c,TX_RESP_Q20d,TX_RESP_Q20e,TX_RESP_Q21
0,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,A,A,A,A,C,B,D,D,B,C
1,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,B,C,C,A,B,D,D,D,C,B
4,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,C,A,.,A,B,*,A,A,D,C
5,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,B,A,B,A,C,*,B,B,B,C
6,2021,1,11,6316600,2,61324549,1,1,1324152,9,...,C,A,A,A,B,B,C,D,D,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2591699,2021,5,53,6322169,1,61397023,1,1,1435004,9,...,B,A,A,A,C,A,C,D,C,C
2591701,2021,5,53,6322169,1,61397023,1,1,1435004,9,...,B,A,A,A,.,.,.,.,.,C
2591702,2021,5,53,6322169,1,61397023,1,1,1435004,9,...,C,A,A,A,C,B,B,A,D,C
2591704,2021,5,53,6322169,1,61397023,1,1,1435004,9,...,C,A,A,A,B,D,B,A,D,D


##### 2023

In [81]:
df_2023_red = df_2023_red[(df_2023_red['IN_PROFICIENCIA_MT'] == '1') & (df_2023_red['IN_PUBLICA'] == '1')].copy()
df_2023_red

,ID_SAEB,ID_REGIAO,ID_UF,ID_MUNICIPIO,ID_AREA,ID_ESCOLA,IN_PUBLICA,ID_LOCALIZACAO,ID_TURMA,ID_SERIE,...,TX_RESP_Q17,TX_RESP_Q18,TX_RESP_Q19,TX_RESP_Q20,TX_RESP_Q21a,TX_RESP_Q21b,TX_RESP_Q21c,TX_RESP_Q21d,TX_RESP_Q21e,TX_RESP_Q24
0,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,C,A,A,A,A,A,D,D,D,B
3,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,C,A,A,A,.,.,.,.,D,A
7,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,B,A,B,A,B,A,D,A,C,A
9,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,B,A,B,A,B,A,C,D,B,C
11,2023,1,11,6322170,2,61400934,1,1,1704202,9,...,B,A,A,A,B,A,D,A,C,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2502898,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,C,C,A,A,C,B,B,A,D,C
2502900,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,C,A,A,A,B,A,B,A,D,D
2502902,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,C,A,A,A,B,C,A,A,D,C
2502903,2023,5,53,6327738,1,61470349,1,1,1719813,9,...,A,C,A,A,C,D,D,C,D,C


#### 3. Padronização dos datasets
Após a etapa 1 descrita acima, foram mantidas somente as variáveis (colunas) comuns a todos os anos. No entanto, entre um ano e outro são observadas diferenças na ordem em que algumas colunas aparecem e na apresentação de algumas alternativas. Essas questões devem ser tratadas para que não comprometam as análises subsequentes.
O primeiro passo para ordenar as colunas é definir um padrão de nomes a ser seguido pelas colunas em todos os datasets:

In [82]:
colunas_padrao = ["ANO","REGIAO","UF","MUNICIPIO","AREA","ID_ESCOLA","PUBLICA","LOCALIZACAO","ID_TURMA","SERIE","ID_ALUNO","SITUACAO_CENSO",
                  "PROFICIENCIA","PESO","PROFICIENCIA_MT","ERRO_PADRAO_MT","PROFICIENCIA_SAEB","ERRO_PADRAO_MT_SAEB","LINGUA_FALADA",
                  "COR/RACA","MORADOR_MAE","MORADOR_PAI","MORADOR_OUTROS","ESCOL_MAE","ESCOL_PAI","FREQUENCIA_CONVERSA","FREQUENCIA_INCENTIVO_ESTUDO",
                  "FREQUENCIA_INCENTIVO_TAREFA","FREQUENCIA_INCENTIVO_AULA","FREQUENCIA_REUNIOES","ASFALTO/CALCAMENTO","AGUA_TRATADA","ILUMINACAO",
                  "QTD_GELADEIRA","QTD_COMPUTADOR","QTD_QUARTO","QTD_TELEVISAO","QTD_BANHEIRO","QTD_CARRO","TV_A_CABO","WI-FI","QUARTO","MESA","GARAGEM",
                  "MICROONDAS","ASPIRADOR","MAQUINA","FREEZER","TEMPO_ATE_ESCOLA","LOCOMOCAO_ATE_ESCOLA","IDADE_INTROD_ESC","TIPO_ESCOLA","REPROVACAO",
                  "ABANDONO","TEMPO_LAZER","TEMPO_CURSOS","TEMPO_TRAB_DOMES","TEMPO_ESTUDO","TEMPO_TRABALHO","POS_EF"]

Na sequência, para cada dataset, são mapeados os nomes atuais para os nomes padrão, e renomeadas as colunas:

##### 2019:

In [83]:
mapa_df_2019_red = {
    "ID_SAEB": "ANO",
    "ID_REGIAO": "REGIAO",
    "ID_UF": "UF",
    "ID_MUNICIPIO": "MUNICIPIO",
    "ID_AREA": "AREA",
    "ID_ESCOLA": "ID_ESCOLA",
    "IN_PUBLICA": "PUBLICA",
    "ID_LOCALIZACAO": "LOCALIZACAO",
    "ID_TURMA": "ID_TURMA",
    "ID_SERIE": "SERIE",
    "ID_ALUNO": "ID_ALUNO",
    "IN_SITUACAO_CENSO": "SITUACAO_CENSO",
    "IN_PROFICIENCIA_MT": "PROFICIENCIA",
    "PESO_ALUNO_MT": "PESO",
    "PROFICIENCIA_MT": "PROFICIENCIA_MT",
    "ERRO_PADRAO_MT": "ERRO_PADRAO_MT",
    "PROFICIENCIA_MT_SAEB": "PROFICIENCIA_SAEB",
    "ERRO_PADRAO_MT_SAEB": "ERRO_PADRAO_MT_SAEB",    
    "TX_RESP_Q001": "LINGUA_FALADA",
    "TX_RESP_Q002": "COR/RACA",
    "TX_RESP_Q003A": "MORADOR_MAE",
    "TX_RESP_Q003B": "MORADOR_PAI",
    "TX_RESP_Q003E": "MORADOR_OUTROS",
    "TX_RESP_Q004": "ESCOL_MAE",
    "TX_RESP_Q005": "ESCOL_PAI",
    "TX_RESP_Q006A": "FREQUENCIA_CONVERSA",
    "TX_RESP_Q006B": "FREQUENCIA_INCENTIVO_ESTUDO",
    "TX_RESP_Q006C": "FREQUENCIA_INCENTIVO_TAREFA",
    "TX_RESP_Q006D": "FREQUENCIA_INCENTIVO_AULA",
    "TX_RESP_Q006E": "FREQUENCIA_REUNIOES",
    "TX_RESP_Q008A": "ASFALTO/CALCAMENTO",
    "TX_RESP_Q008B": "AGUA_TRATADA",
    "TX_RESP_Q008C": "ILUMINACAO",
    "TX_RESP_Q009A": "QTD_GELADEIRA",
    "TX_RESP_Q009C": "QTD_COMPUTADOR",
    "TX_RESP_Q009D": "QTD_QUARTO",
    "TX_RESP_Q009E": "QTD_TELEVISAO",
    "TX_RESP_Q009F": "QTD_BANHEIRO",
    "TX_RESP_Q009G": "QTD_CARRO",
    "TX_RESP_Q010A": "TV_A_CABO",
    "TX_RESP_Q010B": "WI-FI",
    "TX_RESP_Q010C": "QUARTO",
    "TX_RESP_Q010D": "MESA",
    "TX_RESP_Q010E": "GARAGEM",
    "TX_RESP_Q010F": "MICROONDAS",
    "TX_RESP_Q010G": "ASPIRADOR",
    "TX_RESP_Q010H": "MAQUINA",
    "TX_RESP_Q010I": "FREEZER",
    "TX_RESP_Q011": "TEMPO_ATE_ESCOLA",
    "TX_RESP_Q012": "LOCOMOCAO_ATE_ESCOLA",
    "TX_RESP_Q013": "IDADE_INTROD_ESC",     
    "TX_RESP_Q014": "TIPO_ESCOLA",
    "TX_RESP_Q015": "REPROVACAO",
    "TX_RESP_Q016": "ABANDONO",
    "TX_RESP_Q017A": "TEMPO_LAZER",
    "TX_RESP_Q017B": "TEMPO_CURSOS",
    "TX_RESP_Q017C": "TEMPO_TRAB_DOMES",
    "TX_RESP_Q017D": "TEMPO_ESTUDO",
    "TX_RESP_Q017E": "TEMPO_TRABALHO",
    "TX_RESP_Q019": "POS_EF",
}
df_2019_red = df_2019_red.rename(columns=mapa_df_2019_red)
#df_2019_red

##### 2021:

In [84]:
mapa_df_2021_red = {
    "ID_SAEB": "ANO",
    "ID_REGIAO": "REGIAO",
    "ID_UF": "UF",
    "ID_MUNICIPIO": "MUNICIPIO",
    "ID_AREA": "AREA",
    "ID_ESCOLA": "ID_ESCOLA",
    "IN_PUBLICA": "PUBLICA",
    "ID_LOCALIZACAO": "LOCALIZACAO",
    "ID_TURMA": "ID_TURMA",
    "ID_SERIE": "SERIE",
    "ID_ALUNO": "ID_ALUNO",
    "IN_SITUACAO_CENSO": "SITUACAO_CENSO",
    "IN_PROFICIENCIA_MT": "PROFICIENCIA",
    "PESO_ALUNO_MT": "PESO",
    "PROFICIENCIA_MT": "PROFICIENCIA_MT",
    "ERRO_PADRAO_MT": "ERRO_PADRAO_MT",
    "PROFICIENCIA_MT_SAEB": "PROFICIENCIA_SAEB",
    "ERRO_PADRAO_MT_SAEB": "ERRO_PADRAO_MT_SAEB",     
    "TX_RESP_Q03": "LINGUA_FALADA",
    "TX_RESP_Q04": "COR/RACA",
    "TX_RESP_Q06a": "MORADOR_MAE",
    "TX_RESP_Q06b": "MORADOR_PAI",
    "TX_RESP_Q06e": "MORADOR_OUTROS",    
    "TX_RESP_Q07": "ESCOL_MAE",
    "TX_RESP_Q08": "ESCOL_PAI",
    "TX_RESP_Q09b": "FREQUENCIA_CONVERSA",
    "TX_RESP_Q09c": "FREQUENCIA_INCENTIVO_ESTUDO",
    "TX_RESP_Q09d": "FREQUENCIA_INCENTIVO_TAREFA",
    "TX_RESP_Q09e": "FREQUENCIA_INCENTIVO_AULA",
    "TX_RESP_Q09f": "FREQUENCIA_REUNIOES",    
    "TX_RESP_Q10a": "ASFALTO/CALCAMENTO",
    "TX_RESP_Q10b": "AGUA_TRATADA",
    "TX_RESP_Q10c": "ILUMINACAO",
    "TX_RESP_Q11a": "QTD_GELADEIRA",
    "TX_RESP_Q11c": "QTD_COMPUTADOR",
    "TX_RESP_Q11d": "QTD_QUARTO",
    "TX_RESP_Q11e": "QTD_TELEVISAO",
    "TX_RESP_Q11f": "QTD_BANHEIRO",
    "TX_RESP_Q11g": "QTD_CARRO",
    "TX_RESP_Q12a": "TV_A_CABO",
    "TX_RESP_Q12b": "WI-FI",
    "TX_RESP_Q12c": "QUARTO",
    "TX_RESP_Q12d": "MESA",
    "TX_RESP_Q12e": "MICROONDAS",
    "TX_RESP_Q12f": "ASPIRADOR",
    "TX_RESP_Q12g": "MAQUINA",
    "TX_RESP_Q12h": "FREEZER",
    "TX_RESP_Q12i": "GARAGEM",
    "TX_RESP_Q13": "TEMPO_ATE_ESCOLA",
    "TX_RESP_Q14": "LOCOMOCAO_ATE_ESCOLA",
    "TX_RESP_Q16": "IDADE_INTROD_ESC",     
    "TX_RESP_Q17": "TIPO_ESCOLA",
    "TX_RESP_Q18": "REPROVACAO",
    "TX_RESP_Q19": "ABANDONO",   
    "TX_RESP_Q20a": "TEMPO_ESTUDO",
    "TX_RESP_Q20b": "TEMPO_CURSOS",
    "TX_RESP_Q20c": "TEMPO_TRAB_DOMES",
    "TX_RESP_Q20d": "TEMPO_TRABALHO",
    "TX_RESP_Q20e": "TEMPO_LAZER",
    "TX_RESP_Q21": "POS_EF",
}
df_2021_red = df_2021_red.rename(columns=mapa_df_2021_red)
#df_2021_red

##### 2023:

In [85]:
mapa_df_2023_red = {
    "ID_SAEB": "ANO",
    "ID_REGIAO": "REGIAO",
    "ID_UF": "UF",
    "ID_MUNICIPIO": "MUNICIPIO",
    "ID_AREA": "AREA",
    "ID_ESCOLA": "ID_ESCOLA",
    "IN_PUBLICA": "PUBLICA",
    "ID_LOCALIZACAO": "LOCALIZACAO",
    "ID_TURMA": "ID_TURMA",
    "ID_SERIE": "SERIE",
    "ID_ALUNO": "ID_ALUNO",
    "IN_SITUACAO_CENSO": "SITUACAO_CENSO",
    "IN_PROFICIENCIA_MT": "PROFICIENCIA",
    "PESO_ALUNO_MT": "PESO",
    "PROFICIENCIA_MT": "PROFICIENCIA_MT",
    "ERRO_PADRAO_MT": "ERRO_PADRAO_MT",
    "PROFICIENCIA_MT_SAEB": "PROFICIENCIA_SAEB",
    "ERRO_PADRAO_MT_SAEB": "ERRO_PADRAO_MT_SAEB",
    "TX_RESP_Q03": "LINGUA_FALADA",
    "TX_RESP_Q04": "COR/RACA",
    "TX_RESP_Q07a": "MORADOR_MAE",
    "TX_RESP_Q07b": "MORADOR_PAI",
    "TX_RESP_Q07e": "MORADOR_OUTROS",
    "TX_RESP_Q08": "ESCOL_MAE",
    "TX_RESP_Q09": "ESCOL_PAI",
    "TX_RESP_Q10b": "FREQUENCIA_CONVERSA",
    "TX_RESP_Q10c": "FREQUENCIA_INCENTIVO_ESTUDO",
    "TX_RESP_Q10d": "FREQUENCIA_INCENTIVO_TAREFA",
    "TX_RESP_Q10e": "FREQUENCIA_INCENTIVO_AULA",
    "TX_RESP_Q10f": "FREQUENCIA_REUNIOES", 
    "TX_RESP_Q11a": "ASFALTO/CALCAMENTO",
    "TX_RESP_Q11b": "AGUA_TRATADA",
    "TX_RESP_Q11c": "ILUMINACAO",
    "TX_RESP_Q12a": "QTD_GELADEIRA",
    "TX_RESP_Q12b": "QTD_COMPUTADOR",
    "TX_RESP_Q12c": "QTD_QUARTO",
    "TX_RESP_Q12d": "QTD_TELEVISAO",
    "TX_RESP_Q12e": "QTD_BANHEIRO",
    "TX_RESP_Q12f": "QTD_CARRO",
    "TX_RESP_Q13a": "TV_A_CABO",
    "TX_RESP_Q13b": "WI-FI",
    "TX_RESP_Q13c": "QUARTO",
    "TX_RESP_Q13d": "MESA",
    "TX_RESP_Q13e": "MICROONDAS",
    "TX_RESP_Q13f": "ASPIRADOR",
    "TX_RESP_Q13g": "MAQUINA",
    "TX_RESP_Q13h": "FREEZER",
    "TX_RESP_Q13i": "GARAGEM",
    "TX_RESP_Q14": "TEMPO_ATE_ESCOLA",
    "TX_RESP_Q16": "LOCOMOCAO_ATE_ESCOLA",
    "TX_RESP_Q17": "IDADE_INTROD_ESC",     
    "TX_RESP_Q18": "TIPO_ESCOLA",
    "TX_RESP_Q19": "REPROVACAO",
    "TX_RESP_Q20": "ABANDONO",   
    "TX_RESP_Q21a": "TEMPO_ESTUDO",
    "TX_RESP_Q21b": "TEMPO_CURSOS",
    "TX_RESP_Q21c": "TEMPO_TRAB_DOMES",
    "TX_RESP_Q21d": "TEMPO_TRABALHO",
    "TX_RESP_Q21e": "TEMPO_LAZER",
    "TX_RESP_Q24": "POS_EF",
}
df_2023_red = df_2023_red.rename(columns=mapa_df_2023_red)
#df_2023_red

Por último, a ordem das colunas de todos os datasets é padronizada: 

In [86]:
df_2019_red = df_2019_red.reindex(columns=colunas_padrao)
df_2021_red = df_2021_red.reindex(columns=colunas_padrao)
df_2023_red = df_2023_red.reindex(columns=colunas_padrao)

#### 4. Unificação das bases dos diferentes anos
Realizar a união de bases de dados em uma análise multianual é fundamental para garantir uma visão mais completa, consistente e comparável ao longo do tempo. Isso permite unir as informações dos diferentes anos em uma única estrutura, facilitando sua exploração.

In [87]:
gc.collect()

!pip install fastparquet

df_unif = pd.concat([df_2019_red, df_2021_red, df_2023_red], ignore_index=True)
processed_dir = r"C:\Users\emill\Downloads\TCC\processed"
os.makedirs(processed_dir, exist_ok=True)

out_path = os.path.join(processed_dir, "BASE_UNIF.parquet")

df_unif.to_parquet(out_path, index=False, engine="fastparquet", compression="gzip")

Defaulting to user installation because normal site-packages is not writeable


#### 5. Remoção de duplicatas
Como cada linha do dataset representa exclusivamente um aluno, foi realizada uma verificação para identificar possíveis registros repetidos utilizando o comando abaixo. Esse procedimento garante que não haja inconsistências por múltiplas ocorrências de um mesmo aluno.

In [88]:
df_unif.duplicated().sum()

np.int64(0)

Como o resultado retornado foi 0, não existem registros repetidos no conjunto de dados, nenhuma linha precisou ser removida e sua estrutura permaneceu inalterada.

#### 6. Tratamento de erros e inconsistências de codificação
A qualidade de um conjunto de dados é fundamental para garantir a validade das análises. Por isso, as variáveis do conjunto de dados que contêm valores categóricos foram verificadas quanto a possíveis erros relacionados à codificação, como formatação inconsistente dos valores. 

##### Função UNIQUE
Validando se as possíveis respostas tem formatação diferente.

In [89]:
df_unif['ANO'].unique()

<StringArray>
['2019', '2021', '2023']
Length: 3, dtype: str

In [90]:
df_unif['REGIAO'].unique()

<StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str

In [91]:
df_unif['UF'].unique()

<StringArray>
['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26',
 '27', '28', '29', '31', '32', '33', '35', '41', '42', '43', '50', '51', '52',
 '53']
Length: 27, dtype: str

In [92]:
df_unif['MUNICIPIO'].unique()

<StringArray>
['6311030', '6311031', '6311032', '6311033', '6311034', '6311035', '6311036',
 '6311037', '6311038', '6311039',
 ...
 '6327729', '6327730', '6327731', '6327732', '6327733', '6327734', '6327735',
 '6327736', '6327737', '6327738']
Length: 16604, dtype: str

In [93]:
df_unif['AREA'].unique()

<StringArray>
['2', '1']
Length: 2, dtype: str

In [94]:
df_unif['ID_ESCOLA'].unique()

<StringArray>
['61255632', '61260610', '61281754', '61295480', '61311224', '61312614',
 '61321894', '61256082', '61256332', '61257314',
 ...
 '61470196', '61470203', '61470206', '61470208', '61470213', '61470214',
 '61470215', '61470330', '61470338', '61470349']
Length: 109760, dtype: str

In [95]:
df_unif['UF'].unique()

<StringArray>
['11', '12', '13', '14', '15', '16', '17', '21', '22', '23', '24', '25', '26',
 '27', '28', '29', '31', '32', '33', '35', '41', '42', '43', '50', '51', '52',
 '53']
Length: 27, dtype: str

In [96]:
df_unif['LOCALIZACAO'].unique()

<StringArray>
['1', '2']
Length: 2, dtype: str

In [97]:
df_unif['ID_TURMA'].unique()

<StringArray>
['1030927', '1209121', '1030965',  '981213',  '996797', '1000764', '1008480',
 '1066545', '1101590', '1124709',
 ...
 '1672716', '1649024', '1680483', '1727662', '1530756', '1546548', '1657056',
 '1664979', '1696233', '1719813']
Length: 245274, dtype: str

In [98]:
df_unif['SERIE'].unique()

<StringArray>
['9']
Length: 1, dtype: str

In [99]:
df_unif['ID_ALUNO'].unique()

<StringArray>
['34874910', '34874911', '34874912', '34874913', '34874914', '34874916',
 '34874917', '34874918', '34874919', '34874920',
 ...
 '56417941', '56417942', '56417943', '56417944', '56417945', '56417946',
 '56417948', '56417950', '56417951', '56418025']
Length: 5850766, dtype: str

In [100]:
df_unif['SITUACAO_CENSO'].unique()

<StringArray>
['1', '0']
Length: 2, dtype: str

In [101]:
df_unif['PROFICIENCIA'].unique()

<StringArray>
['1']
Length: 1, dtype: str

A verificação das variáveis demonstrou que não há inconsistências de codificação, todas os valores das colunas analisadas estão de acordo com o esperado. Isso garante maior confiabilidade aos dados e elimina a necessidade de tratamentos adicionais nessas variáveis.

#### 7. Valores nulos

As informações gerais sobre o conjunto de dados foram exibidas novamente para determinar as colunas que continham valores nulos (ou seja, aquelas cuja contagem de valores não nulos não é igual ao número de observações).

In [102]:
df_unif.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 5850766 entries, 0 to 5850765
Data columns (total 60 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   ANO                          5850766 non-null  str  
 1   REGIAO                       5850766 non-null  str  
 2   UF                           5850766 non-null  str  
 3   MUNICIPIO                    5850766 non-null  str  
 4   AREA                         5850766 non-null  str  
 5   ID_ESCOLA                    5850766 non-null  str  
 6   PUBLICA                      5850766 non-null  str  
 7   LOCALIZACAO                  5850766 non-null  str  
 8   ID_TURMA                     5850766 non-null  str  
 9   SERIE                        5850766 non-null  str  
 10  ID_ALUNO                     5850766 non-null  str  
 11  SITUACAO_CENSO               5850766 non-null  str  
 12  PROFICIENCIA                 5850766 non-null  str  
 13  PESO                   

In [103]:
df_unif.query('PESO != PESO')

,ANO,REGIAO,UF,MUNICIPIO,AREA,ID_ESCOLA,PUBLICA,LOCALIZACAO,ID_TURMA,SERIE,...,IDADE_INTROD_ESC,TIPO_ESCOLA,REPROVACAO,ABANDONO,TEMPO_LAZER,TEMPO_CURSOS,TEMPO_TRAB_DOMES,TEMPO_ESTUDO,TEMPO_TRABALHO,POS_EF
1304,2019,1,11,6311031,2,61299589,1,1,1054734,9,...,A,A,A,A,C,A,C,C,A,C
1429,2019,1,11,6311031,2,61300421,1,2,1101956,9,...,B,A,A,A,C,A,C,C,C,C
4517,2019,1,11,6311040,2,61271266,1,1,1086067,9,...,C,A,A,A,D,A,B,A,A,C
5018,2019,1,11,6311040,2,61303627,1,1,1190296,9,...,B,A,A,A,D,A,A,B,A,C
8308,2019,1,11,6311046,1,61253281,1,1,1170877,9,...,C,A,A,B,D,B,B,B,D,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5843627,2023,5,53,6327738,1,61463274,1,1,1546518,9,...,.,.,.,.,.,.,.,.,.,A
5846289,2023,5,53,6327738,1,61466343,1,1,1688309,9,...,A,A,A,A,B,B,C,B,C,C
5846662,2023,5,53,6327738,1,61466354,1,1,1719744,9,...,A,A,A,A,D,A,B,C,D,C
5847738,2023,5,53,6327738,1,61469533,1,1,1625544,9,...,B,C,B,A,D,B,C,C,A,A


#### 8. Tipos de Dados incorretos

#### 9. Reinicialização do índice